# TI-AI-09 · Scalarized MORL route policy (PIC A)

One preference-conditioned policy (Graph Encoder, Preference Encoder, Actor-Critic) trained on `route-env-1` with the
weighted reward $r = \alpha\, r_{time} + \beta\, r_{exposure}$. Each episode draws $\alpha \sim U[0, 1]$, $\beta = 1 - \alpha$.

Runs locally on CPU. The code lives in `src/backend/route_model/morl/` and this notebook only calls it, so the
terminal commands in [`docs/experiments/TI-AI-09/scalarized/README.md`](../../docs/experiments/TI-AI-09/scalarized/README.md)
do exactly the same thing. **Prefer the terminal for the full run:** it outlives a closed notebook tab.

Build the fields first (step 2 of that README). Run this notebook with `CONFIG_NAME` on the smoke config (about 3
minutes) before the real one.

## 1 · Settings

In [ ]:
import json, os, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/backend/route_model").is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

CONFIG_NAME = "scalarized-bike-smoke"   # "scalarized-bike-v1" for the real run
RESUME_RUN = None                       # a run folder to continue after a stop, e.g. "src/backend/route_model/runs/scalarized/2026..."
EXISTING_RUN = None                     # a finished run folder, to skip training and only report on it
MAX_HOURS = None                        # e.g. 6: stop cleanly after this long; continue later with RESUME_RUN

CONFIG_PATH = Path(f"src/backend/route_model/morl/configs/{CONFIG_NAME}.json")
FIELDS = Path("src/backend/route_model/runs/fields")
OUT = Path("src/backend/route_model/runs/scalarized")
assert (FIELDS / "manifest.json").exists(), "No fields yet: build them first (step 2 of the README)"

## 2 · Train

Warm start (imitating the exact route for each episode's preference), then PPO on the scalarized reward. Validation
runs every `schedule.eval_every` updates, and `checkpoints/best.pt` is the best mean return over the preference grid on
real validation fields. Interrupting the kernel is safe: the run records why it stopped and can continue from `last.pt`.

In [ ]:
from src.backend.route_model.morl.scalarized import train_scalarized
from src.backend.route_model.morl.train import load_config

if EXISTING_RUN:
    RUN_DIR = Path(EXISTING_RUN)
else:
    run = train_scalarized(load_config(CONFIG_PATH), FIELDS, None if RESUME_RUN else OUT, resume=RESUME_RUN,
                           max_seconds=MAX_HOURS * 3600 if MAX_HOURS else None)
    RUN_DIR = run.dir
manifest = json.loads((RUN_DIR / "manifest.json").read_text(encoding="utf-8"))
print(RUN_DIR)
print(manifest["status"], "|", manifest["stop_reason"])
print(json.dumps(manifest["best"], indent=2))

## 3 · Reload check

Checks the checkpoint against the SHA-256 in the manifest, loads it twice, and compares greedy routes on every validation
OD pair at every selection preference (acceptance criterion 1).

In [ ]:
from src.backend.route_model.morl.scalarized import verify

report = verify(RUN_DIR, FIELDS)
print(json.dumps(report, indent=2))
assert report["checkpoint_matches_manifest"] and report["reload"]["identical"], "The checkpoint does not reproduce its routes"

## 4 · Curves, summary and evidence

In [ ]:
from IPython.display import Image, Markdown, display
from src.backend.route_model.morl.scalarized import export_evidence

evidence = export_evidence(RUN_DIR)
display(Image(str(RUN_DIR / "training_curves.png")))
display(Markdown((RUN_DIR / "summary.md").read_text(encoding="utf-8")))
print("Commit this folder:", evidence)